# "Covid-19 ~ India"
> "Some charts exploring Covid-19 India Data"

- toc: false 
- badges: false
- comments: true
- categories: [ai-in-society]
- image: images/coronavirus.jpg
- permalink: /covid-19/

In [ ]:
#hide
from IPython.display import HTML, Javascript, display
from string import Template
import numpy as np
import pandas as pd
import json
import math
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
import time
import requests

In [ ]:
#hide
mpl.rcParams['figure.dpi'] = 100
display(Javascript("require.config({paths: {d3: 'https://d3js.org/d3.v5.min.js'}});"));

In [16]:
#hide
"""Simple script to print HTML style markdown text in a notebook"""
from IPython.display import Markdown, display

class BeautifulText():
    """
    Display HTML format Markdown text in the notebook. Useful for highlighting important notes in
    the notebook.
    """
    def __init__(self, color: str = 'black', font_family: str = 'Arial',
                 font_weight: str = 'normal', font_size: int = 16, font_style: str = 'normal',
                 background_color: str = 'none', letter_spacing: int = 1, line_height: float = 0.8,
                 word_spacing: int = 1, text_decoration: str = 'none', text_shadow: str = 'none'):
        """
        Initialise with HTML style elements.
        Parameters
        ----------
        color : str
            Color of the markdown text. Defaults to `black`.
        font_family : str
            `font-family` html attribute. Defaults to `serif`.
        font_weight : str
            `font-weight` html attribute. Defaults to `normal`.
        font_size : int
            `font-size` html attribute. Defaults to `16`px. (Don't include `px` in the argument.)
        font_style : str
            `font-style` html attribute. Defaults to `normal`.
        background_color : str
            `background-color` html attribute. Defaults to `none`
        letter_spacing : int
            `letter-spacing` html attribute. Defaults to `1`px.
            (Don't include `px` in the argument.)
        line_height : float
            `line-height` html attribute. Defaults to `0.8`px. (Don't include `px` in the argument.)
        word_spacing : int
            `word-spacing` html attribute. Defaults to `1`px. (Don't include `px` in the argument.)
        text-decoration : str
            `text-decoration` html attribute. Defaults to `none`.
        text-shadow : str
            `text-shadow` html attribute. Defaults to `none`.
        """
        self.color = color
        self.font_family = font_family
        self.font_weight = font_weight
        self.font_size = font_size
        self.font_style = font_style
        self.background_color = background_color
        self.letter_spacing = letter_spacing
        self.line_height = line_height
        self.word_spacing = word_spacing
        self.text_decoration = text_decoration
        self.text_shadow = text_shadow
    def printbeautiful(self, text: str) -> None:
        """Applies the html attributes to `text` and prints it as MarkDown."""
        beautifultext = f"""<span style='color: {self.color};
                        font-family: {self.font_family};
                        font-weight: {self.font_weight};
                        font-size: {self.font_size}px;
                        font-style: {self.font_style};
                        text-decoration: {self.text_decoration};
                        letter-spacing: {self.letter_spacing}px;
                        line-height: {self.line_height}px;
                        word-spacing: {self.word_spacing}px;
                        text-shadow: {self.text_shadow};
                        background-color: {self.background_color};'>
                        {text}
                        </span>
                        """
        display(Markdown(beautifultext))


In [13]:
#hide
commentary = BeautifulText(font_family="Calligraffitti", font_size=12, line_height=0.5)

In [ ]:
#hide
css_text = '''
.wrapper {
    position: relative;
}

.line {
    fill: none;
    stroke: #FF9933;
    stroke-width: 2;
}

.y-axis-label {
    fill: black;
    font-size: 1.4em;
    text-anchor: middle;
    transform: rotate(-90deg);
}


.main-listening-rect {
    fill: transparent;
}

.listening-rect {
    fill: transparent;
}

body {
    display: flex;
    justify-content: center;
    padding: 5em 1em;
    font-family: sans-serif;
}

.main-tooltip {
    opacity: 0;
    position: absolute;
    top: -14px;
    left: 0;
    padding: 0.6em 1em;
    background: #fff;
    text-align: center;
    line-height: 1.4em;
    font-size: 0.9em;
    border: 1px solid #ddd;
    z-index: 10;
    transition: all 0.1s ease-out;
    pointer-events: none;
}

.tooltip {
    opacity: 0;
    position: absolute;
    top: -14px;
    left: 0;
    padding: 0.6em 1em;
    background: #fff;
    text-align: center;
    line-height: 1.4em;
    font-size: 0.9em;
    border: 1px solid #ddd;
    z-index: 10;
    transition: all 0.1s ease-out;
    pointer-events: none;
}

.main-tooltip:before {
    content: '';
    position: absolute;
    bottom: 0;
    left: 50%;
    width: 12px;
    height: 12px;
    background: white;
    border: 1px solid #ddd;
    border-top-color: transparent;
    border-left-color: transparent;
    transform: translate(-50%, 50%) rotate(45deg);
    transform-origin: center center;
    z-index: 10;
}

.tooltip:before {
    content: '';
    position: absolute;
    bottom: 0;
    left: 50%;
    width: 12px;
    height: 12px;
    background: white;
    border: 1px solid #ddd;
    border-top-color: transparent;
    border-left-color: transparent;
    transform: translate(-50%, 50%) rotate(45deg);
    transform-origin: center center;
    z-index: 10;
}

.main-tooltip-date {
    margin-bottom: 0.2em;
    font-weight: 600;
    font-size: 1.1em;
    line-height: 1.4em;
}

.tooltip-date {
    margin-bottom: 0.2em;
    font-weight: 600;
    font-size: 1.1em;
    line-height: 1.4em;
}

.rate-wrapper {
    position: relative;
}
    
.rate-line {
    fill: none;
    stroke: #8B0000;
    stroke-width: 2;
}

.summary-wrapper {
    position: relative;
}

.summary-confirmed{
    fill: #8B0000;

}

.confirmed-badge{
    fill : white;
    font-size: 13pt;
    font-weight: 700;
    text-anchor: middle;
    font-family: Arial, Helvetica, sans-serif;
}

.confirmed-badge-value{
    fill : white;
    font-size: 38pt;
    font-weight: 700;
    text-anchor: middle;;
}

.summary-active{
    fill: #FF8C00;

}

.active-badge{
    fill : white;
    font-size: 13pt;
    font-weight: 700;
    text-anchor: middle;
    font-family: Arial, Helvetica, sans-serif;
}

.active-badge-value{
    fill : white;
    font-size: 38pt;
    font-weight: 700;
    text-anchor: middle;;
}

.summary-recovered{
    fill: #006400;

}

.recovered-badge{
    fill : white;
    font-size: 13pt;
    font-weight: 700;
    text-anchor: middle;
    font-family: Arial, Helvetica, sans-serif;
}

.recovered-badge-value{
    fill : white;
    font-size: 38pt;
    font-weight: 700;
    text-anchor: middle;;
}

.summary-deceased{
    fill: #A9A9A9;

}

.deceased-badge{
    fill : white;
    font-size: 13pt;
    font-weight: 700;
    text-anchor: middle;
    font-family: Arial, Helvetica, sans-serif;
}

.deceased-badge-value{
    fill : white;
    font-size: 38pt;
    font-weight: 700;
    text-anchor: middle;;
}

'''

# Summary

In [ ]:
#hide
official_latest_url = "https://api.rootnet.in/covid19-in/stats/latest"
latest_raw = requests.get(official_latest_url).text
latest = json.loads(latest_raw)['data']

In [ ]:
#hide
confirmed_count = latest['summary']['total']
recovered_count = latest['summary']['discharged']
deceased_count = latest['summary']['deaths']
active_count = confirmed_count - (recovered_count + deceased_count)

In [ ]:
#hide
summary_js = f"""
let confirmed = {confirmed_count}
let recovered = {recovered_count}
let deceased = {deceased_count}
let active = {active_count}
"""

In [ ]:
#hide
summary_html_temp = Template('''
    <script src = "https://d3js.org/d3.v5.min.js"></script> 
    <style scoped>
        $css_text
    </style>
    <div id="summary-wrapper" class="summary-wrapper">
        </div>    
    <script>
    $summary_data
    $d3_script_summary
    </script>
''')

In [ ]:
#hide
d3_script_summary = '''
function drawSummaryChart() {

  let dimensions = {
    width: 750,
    height: 100,
  }

  // 3. Draw canvas

  const wrapper = d3.select("#summary-wrapper")
    .append("svg")
      .attr("width", dimensions.width)
      .attr("height", dimensions.height)

  const confirmed_badge = wrapper.append("rect")
    .attr("class", "summary-confirmed")
    .attr("width", "20%")
    .attr("height", "100%")
    .attr("rx", 8)
   

  const active_badge = wrapper.append("rect")
    .attr("class", "summary-active")
    .attr("x", (0.25 * dimensions.width))
    .attr("width", "20%")
    .attr("height", "100%")
    .attr("rx", 8)
   

  const recovered_badge = wrapper.append("rect")
    .attr("class", "summary-recovered")
    .attr("x", (0.50 * dimensions.width))
    .attr("width", "20%")
    .attr("height", "100%")
    .attr("rx", 8)
    

  const deceased_badge = wrapper.append("rect")
    .attr("class", "summary-deceased")
    .attr("x", (0.75 * dimensions.width))
    .attr("width", "20%")
    .attr("height", "100%")
    .attr("rx", 8)

  wrapper.append("text")
  .attr("class", "confirmed-badge")
  .attr("x", (0.2 * dimensions.width)/ 2)
  .attr("y", (0.8 * dimensions.height))
  .text("Confirmed")

  wrapper.append("text")
  .attr("class", "confirmed-badge-value")
  .attr("x", (0.2 * dimensions.width)/ 2)
  .attr("y", (0.5 * dimensions.height))
  .text(`${confirmed}`)

  wrapper.append("text")
  .attr("class", "active-badge")
  .attr("x", (0.25 * dimensions.width) + (0.2 * dimensions.width)/ 2)
  .attr("y", (0.8 * dimensions.height))
  .text("Active")

  wrapper.append("text")
  .attr("class", "active-badge-value")
  .attr("x", (0.25 * dimensions.width) +  (0.2 * dimensions.width)/ 2)
  .attr("y", (0.5 * dimensions.height))
  .text(`${active}`)

  wrapper.append("text")
  .attr("class", "recovered-badge")
  .attr("x", (0.50 * dimensions.width) + (0.2 * dimensions.width)/ 2)
  .attr("y", (0.8 * dimensions.height))
  .text("Recovered")

  wrapper.append("text")
  .attr("class", "active-badge-value")
  .attr("x", (0.50 * dimensions.width) +  (0.2 * dimensions.width)/ 2)
  .attr("y", (0.5 * dimensions.height))
  .text(`${recovered}`)

  wrapper.append("text")
  .attr("class", "deceased-badge")
  .attr("x", (0.75 * dimensions.width) + (0.2 * dimensions.width)/ 2)
  .attr("y", (0.8 * dimensions.height))
  .text("Deceased")

  wrapper.append("text")
  .attr("class", "deceased-badge-value")
  .attr("x", (0.75 * dimensions.width) +  (0.2 * dimensions.width)/ 2)
  .attr("y", (0.5 * dimensions.height))
  .text(`${deceased}`)
      
}
drawSummaryChart();
'''

In [ ]:
#hide
summary_html_text = summary_html_temp.substitute({
    'css_text': css_text,
    'd3_script_summary' : d3_script_summary,
    'summary_data':summary_js
})

In [ ]:
#hide_input
HTML(summary_html_text)

# Covid-19 India Timeline

>Note: The source data API{% fn 1 %} fetches data from March 10, 2020 onwards.

>Tip: Mouse over chart to see data points for a specific day.

In [ ]:
#hide
official_history_url = "https://api.rootnet.in/covid19-in/stats/history" 
data_raw = requests.get(official_history_url).text

In [ ]:
#hide
data = json.loads(data_raw)['data']

In [ ]:
#hide
js_data = f'let dataset = {data}'

In [ ]:
#hide
html_temp = Template('''
    <script src = "https://d3js.org/d3.v5.min.js"></script> 
    <style scoped>
        $css_text
    </style>
    <div id="wrapper" class="wrapper">

            <div id="main-tooltip" class="main-tooltip">
                <div class="main-tooltip-date">
                    <span id="date"></span>
                </div>
                <div class="main-tooltip-confirmed">
                    Confirmed Cases: <span id="main-confirmed"></span>
                </div>
            </div>
        </div>
    <script>
    $dataset
    $d3_script
    </script>
''')

In [ ]:
#hide
d3_script = '''
function drawLineChart() {
  
  main_yAccessor = d => d.summary.total
  const dateParser = d3.timeParse("%Y-%m-%d")
  format = d3.timeFormat("%b-%d")
  const main_xAccessor = d => dateParser(d.day)
  dataset = dataset.sort((a,b) => main_xAccessor(a) - main_xAccessor(b)).slice(0, 100)

  let dimensions = {
    width: 750,
    height: 550,
    margin: {
      top: 20,
      right: 30,
      bottom: 30,
      left: 70,
    },
  }
  dimensions.boundedWidth = dimensions.width - dimensions.margin.left - dimensions.margin.right
  dimensions.boundedHeight = dimensions.height - dimensions.margin.top - dimensions.margin.bottom

  // 3. Draw canvas

  const wrapper = d3.select("#wrapper")
    .append("svg")
      .attr("width", dimensions.width)
      .attr("height", dimensions.height)
      
    wrapper.append("rect")
        .attr("class", "chart-background")
        .attr("width", "100%")
        .attr("height", "100%")
        .attr("fill", "#faebd7")

  const bounds = wrapper.append("g")
      .attr("transform", `translate(${
        dimensions.margin.left
      }, ${
        dimensions.margin.top
      })`)
      
    bounds.append("text")
      .attr("class", "chart-watermark")
      .attr("x", dimensions.width - 180)
      .attr("y", dimensions.height - 60)
      .attr("fill", "grey")
      .style("opacity", 0.5)
      .html("@Alepthoughts")

  bounds.append("defs").append("clipPath")
      .attr("id", "bounds-clip-path")
    .append("rect")
      .attr("width", dimensions.boundedWidth)
      .attr("height", dimensions.boundedHeight)

  const clip = bounds.append("g")
    .attr("clip-path", "url(#bounds-clip-path)")

  // 4. Create scales

  const main_yScale = d3.scaleLinear()
    .domain(d3.extent(dataset, main_yAccessor))
    .range([dimensions.boundedHeight, 0])
    .nice()

  
  const main_xScale = d3.scaleTime()
    .domain(d3.extent(dataset, main_xAccessor))
    .range([0, dimensions.boundedWidth])

  const lineGenerator = d3.line()
    .x(d => main_xScale(main_xAccessor(d)))
    .y(d => main_yScale(main_yAccessor(d)))

  const line = clip.append("path")
      .attr("class", "line")
      .attr("d", lineGenerator(dataset))

  const yAxisGenerator = d3.axisLeft()
      .scale(main_yScale)
      
  
  const yAxis = bounds.append("g")
      .attr("class", "y-axis")
      .call(yAxisGenerator)

  const yAxisLabel = yAxis.append("text")
      .attr("class", "y-axis-label")
      .attr("x", -dimensions.boundedHeight / 2)
      .attr("y", -dimensions.margin.left + 20)
      .html("Confirmed Cases")

      const xAxisGenerator = d3.axisBottom()
      .scale(main_xScale)
      .tickFormat(format)
      .ticks(7)
  
  const xAxis = bounds.append("g")
      .attr("class", "x-axis")
      .style("transform", `translateY(${dimensions.boundedHeight}px)`)
      .call(xAxisGenerator)

  const main_listeningRect = bounds.append("rect")
      .attr("class", "main-listening-rect")
      .attr("width", dimensions.boundedWidth)
      .attr("height", dimensions.boundedHeight)
      .on("mousemove", onMouseMove)
      .on("mouseleave", onMouseLeave)

  const main_tooltip = d3.select("#main-tooltip")
  const main_tooltipCircle = bounds.append("circle")
          .attr("class", "main-tooltip-circle")
          .attr("r", 4)
          .attr("stroke", "#FF9933")
          .attr("fill", "white")
          .attr("stroke-width", 2)
          .style("opacity", 0)

  function onMouseMove() {
    const mousePosition = d3.mouse(this)
    const xhoveredDate = main_xScale.invert(mousePosition[0])
        
    const main_getDistanceFromxhoveredDate = d => Math.abs(main_xAccessor(d) - xhoveredDate)
    const main_closestIndex = d3.scan(dataset, (a, b) => (
    main_getDistanceFromxhoveredDate(a) - main_getDistanceFromxhoveredDate(b)
    ))
    const main_closestDataPoint = dataset[main_closestIndex]
        
    const main_closestXValue = main_xAccessor(main_closestDataPoint)
    const main_closestYValue = main_yAccessor(main_closestDataPoint)
        
    const formatDate = d3.timeFormat("%A %B %-d, %Y")
    main_tooltip.select("#date")
    .text(formatDate(main_closestXValue))

    main_tooltip.select("#main-confirmed")
        .html(main_closestYValue)

    const x = main_xScale(main_closestXValue)
      + dimensions.margin.left
    const y = main_yScale(main_closestYValue)
      + dimensions.margin.top

    main_tooltip.style("transform", `translate(`
      + `calc( -37% + ${x}px),`
      + `calc(-120% + ${y}px)`
      + `)`)

    main_tooltip.style("opacity", 1)

    main_tooltipCircle
        .attr("cx", main_xScale(main_closestXValue))
        .attr("cy", main_yScale(main_closestYValue))
        .style("opacity", 1)
  }
  function onMouseLeave() {
    main_tooltip.style("opacity", 0)

    main_tooltipCircle.style("opacity", 0)
  }
      
}
drawLineChart();
'''

In [ ]:
#hide
html_text = html_temp.substitute({
    'css_text': css_text,
    'd3_script' : d3_script,
    'dataset':js_data
})

In [ ]:
#hide_input
HTML(html_text)

# Growth Rate Over Time

>Tip: Mouse over chart to see data points for a specific day.

In [ ]:
#hide
day = ["2020-03-10"]
growth = [(7/40) * 100]
for i in range(1,len(data)):
    day.append(data[i]['day'])
    growth.append((data[i]['summary']['total'] - data[i-1]['summary']['total'])/data[i-1]['summary']['total'] * 100)

In [ ]:
#hide
rate_data = json.dumps([{"day": day, "growth": growth} for day, growth in zip(day, growth)])

In [ ]:
#hide
rate_data_js = f"let rate_dataset = {rate_data} "

In [ ]:
#hide
rate_html_temp = Template('''
    <script src = "https://d3js.org/d3.v5.min.js"></script> 
    <style scoped>
        $css_text
    </style>
    <div id="rate-wrapper" class="rate-wrapper">

            <div id="tooltip" class="tooltip">
                <div class="tooltip-date">
                    <span id="date"></span>
                </div>
                <div class="tooltip-rate">
                    Rate: <span id="rate"></span>
                </div>
            </div>
        </div>
    <script>
    $rate_dataset
    $d3_rate_script
    </script>
''')

In [ ]:
#hide
d3_rate_script = '''
function drawRateChart() {
  
  yAccessor = d => d.growth
  const dateParser = d3.timeParse("%Y-%m-%d")
  format = d3.timeFormat("%b-%d")
  const xAccessor = d => dateParser(d.day)
  rate_dataset = rate_dataset.sort((a,b) => xAccessor(a) - xAccessor(b)).slice(0, 100)

  let dimensions = {
    width: 750,
    height: 550,
    margin: {
      top: 20,
      right: 30,
      bottom: 30,
      left: 70,
    },
  }
  dimensions.boundedWidth = dimensions.width - dimensions.margin.left - dimensions.margin.right
  dimensions.boundedHeight = dimensions.height - dimensions.margin.top - dimensions.margin.bottom

  // 3. Draw canvas

  const wrapper = d3.select("#rate-wrapper")
    .append("svg")
      .attr("width", dimensions.width)
      .attr("height", dimensions.height)

  wrapper.append("rect")
    .attr("class", "background")
    .attr("width", "100%")
    .attr("height", "100%")
    .attr("fill", "#faebd7")

  const bounds = wrapper.append("g")
      .attr("transform", `translate(${
        dimensions.margin.left
      }, ${
        dimensions.margin.top
      })`)

  

  // 4. Create scales

  const yScale = d3.scaleLinear()
    .domain(d3.extent(rate_dataset, yAccessor))
    .range([dimensions.boundedHeight, 0])
    .nice()

  const LT5Placement = yScale(5)
  const LT5 = bounds.append("rect")
        .attr("x", 0)
        .attr("width", dimensions.boundedWidth)
        .attr("y", LT5Placement)
        .attr("height", dimensions.boundedHeight
          - LT5Placement)
        .attr("fill", "#dcd7fa")

  bounds.append("text")
        .attr("class", "watermark")
        .attr("x", dimensions.width - 180)
        .attr("y", dimensions.height - 60)
        .attr("fill", "grey")
        .style("opacity", 0.5)
        .html("@Alepthoughts")
  
  bounds.append("defs").append("clipPath")
        .attr("id", "bounds-clip-path")
      .append("rect")
        .attr("width", dimensions.boundedWidth)
        .attr("height", dimensions.boundedHeight)
  
    const clip = bounds.append("g")
      .attr("clip-path", "url(#bounds-clip-path)")
  
  const xScale = d3.scaleTime()
    .domain(d3.extent(rate_dataset, xAccessor))
    .range([0, dimensions.boundedWidth])
    .nice()
    

  const lineGenerator = d3.line()
    .x(d => xScale(xAccessor(d)))
    .y(d => yScale(yAccessor(d)))
    .curve(d3.curveMonotoneX)

  const line = clip.append("path")
      .attr("class", "rate-line")
      .attr("d", lineGenerator(rate_dataset))


  const yAxisGenerator = d3.axisLeft()
      .scale(yScale)
      
  
  const yAxis = bounds.append("g")
      .attr("class", "y-axis")
      .call(yAxisGenerator)

  const yAxisLabel = yAxis.append("text")
      .attr("class", "y-axis-label")
      .attr("x", -dimensions.boundedHeight / 2)
      .attr("y", -dimensions.margin.left + 20)
      .html("Growth Rate in (%)")

      const xAxisGenerator = d3.axisBottom()
      .scale(xScale)
      .tickFormat(format)
      .ticks(d3.timeSunday)
      
    
  
  const xAxis = bounds.append("g")
      .attr("class", "x-axis")
      .style("transform", `translateY(${dimensions.boundedHeight}px)`)
      .call(xAxisGenerator)

  const listeningRect = bounds.append("rect")
      .attr("class", "listening-rect")
      .attr("width", dimensions.boundedWidth)
      .attr("height", dimensions.boundedHeight)
      .on("mousemove", onMouseMove)
      .on("mouseleave", onMouseLeave)

  const tooltip = d3.select("#tooltip")
  const tooltipCircle = bounds.append("circle")
          .attr("class", "tooltip-circle")
          .attr("r", 4)
          .attr("stroke", "#8B0000")
          .attr("fill", "white")
          .attr("stroke-width", 2)
          .style("opacity", 0)

  const lockdown = bounds.append("circle")
            .attr("cx", xScale(dateParser("2020-03-25")))
            .attr("cy", yScale(16.76300578034682))
            .attr("r", 4)
            .attr("fill", "#FF0D86")
            .attr("tabindex", "0")

  const lockdownLabel = bounds.append("text")
            .attr("class", "lockdown-annotation")
            .attr("x", xScale(dateParser("2020-03-25")))
            .attr("y", yScale(16.76300578034682) -10)
            .attr("fill", "#FF0D86")
            .attr("font-size", "0.7em")
            .attr("font-family", "Bradley Hand")
            .attr("text-anchor", "middle")
            .text("Lockdown")
            
      
  

  function onMouseMove() {
    const mousePosition = d3.mouse(this)
    const hoveredDate = xScale.invert(mousePosition[0])
        
    const getDistanceFromHoveredDate = d => Math.abs(xAccessor(d) - hoveredDate)
    const closestIndex = d3.scan(rate_dataset, (a, b) => (
    getDistanceFromHoveredDate(a) - getDistanceFromHoveredDate(b)
    ))
    const closestDataPoint = rate_dataset[closestIndex]
        
    const closestXValue = xAccessor(closestDataPoint)
    const closestYValue = yAccessor(closestDataPoint)
        
    const formatDate = d3.timeFormat("%A %B %-d, %Y")
    tooltip.select("#date")
    .text(formatDate(closestXValue))

    const formatRate = d => `${d3.format(".2f")(d)}%`
    tooltip.select("#rate")
        .html(formatRate(closestYValue))

    const x = xScale(closestXValue)
      + dimensions.margin.left
    const y = yScale(closestYValue)
      + dimensions.margin.top

    tooltip.style("transform", `translate(`
      + `calc(-50% + ${x}px),`
      + `calc(-120% + ${y}px)`
      + `)`)

    tooltip.style("opacity", 1)

    tooltipCircle
        .attr("cx", xScale(closestXValue))
        .attr("cy", yScale(closestYValue))
        .style("opacity", 1)
  }
  function onMouseLeave() {
    tooltip.style("opacity", 0)

    tooltipCircle.style("opacity", 0)
  } 
      
}
drawRateChart();
'''

In [ ]:
#hide
html_rate_text = rate_html_temp.substitute({
    'css_text': css_text,
    'd3_rate_script' : d3_rate_script,
    'rate_dataset':rate_data_js
})

In [ ]:
#hide_input
HTML(html_rate_text)

In [17]:
#hide_input
commentary.printbeautiful("The above chart tracks % change in the number of cases on a given day.") 
commentary.printbeautiful("The shaded region shows the number of days when the cofirmed cases grew by less than 5%.")
commentary.printbeautiful("The pink datapoint marks the beginning of the government initiated lockdown of 1.3 billion Indian population.")

<span style='color: black;
                        font-family: Calligraffitti;
                        font-weight: normal;
                        font-size: 12px;
                        font-style: normal;
                        text-decoration: none;
                        letter-spacing: 1px;
                        line-height: 0.5px;
                        word-spacing: 1px;
                        text-shadow: none;
                        background-color: none;'>
                        The above chart tracks % change in the number of cases on a given day.
                        </span>
                        

<span style='color: black;
                        font-family: Calligraffitti;
                        font-weight: normal;
                        font-size: 12px;
                        font-style: normal;
                        text-decoration: none;
                        letter-spacing: 1px;
                        line-height: 0.5px;
                        word-spacing: 1px;
                        text-shadow: none;
                        background-color: none;'>
                        The shaded region shows the number of days when the cofirmed cases grew by less than 5%.
                        </span>
                        

<span style='color: black;
                        font-family: Calligraffitti;
                        font-weight: normal;
                        font-size: 12px;
                        font-style: normal;
                        text-decoration: none;
                        letter-spacing: 1px;
                        line-height: 0.5px;
                        word-spacing: 1px;
                        text-shadow: none;
                        background-color: none;'>
                        The pink datapoint marks the beginning of the government initiated lockdown of 1.3 billion Indian population.
                        </span>
                        

{{ 'The official Ministry of Health and Familiy Welfare API can be found [here](https://api.rootnet.in/)